# Stress Score Prediction V2

V1의 누수 방지 전처리와 LightGBM을 유지하면서, 각 행 내부 정보만으로 계산한 파생변수를 추가한 버전입니다.


## 0. Import & Data Load


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score


In [12]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (3000, 18)
Test shape: (3000, 17)


## 1. Data Split


In [13]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()


## 2. Row-level Feature Engineering


In [14]:
def add_features(df):
    """다른 행이나 데이터셋 통계를 사용하지 않는 행 단위 파생변수 생성 함수."""
    df = df.copy()

    height_m = df['height'] / 100

    df['bmi'] = df['weight'] / (height_m ** 2)
    df['pulse_pressure'] = (
        df['systolic_blood_pressure']
        - df['diastolic_blood_pressure']
    )
    df['mean_arterial_pressure'] = (
        df['systolic_blood_pressure']
        + 2 * df['diastolic_blood_pressure']
    ) / 3
    df['blood_pressure_ratio'] = (
        df['systolic_blood_pressure']
        / (df['diastolic_blood_pressure'] + 1e-6)
    )
    df['cholesterol_glucose_ratio'] = (
        df['cholesterol']
        / (df['glucose'] + 1e-6)
    )
    df['missing_count'] = df.isna().sum(axis=1)

    return df


X_v2 = add_features(X)
X_test_v2 = add_features(X_test)

print('V1 shape:', X.shape)
print('V2 shape:', X_v2.shape)


V1 shape: (3000, 16)
V2 shape: (3000, 22)


## 3. Leakage-safe Preprocessing


In [15]:
categorical_cols_v2 = X_v2.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols_v2 = X_v2.select_dtypes(
    include=['number']
).columns.tolist()

print('범주형컬럼:', categorical_cols_v2)
print('순서형컬럼:', numerical_cols_v2)


범주형컬럼: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
순서형컬럼: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'cholesterol_glucose_ratio', 'missing_count']


In [16]:
categorical_pipeline_v2 = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='constant',
            fill_value='missing'
        )
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True
        )
    )
])

numerical_pipeline_v2 = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='median',
            add_indicator=True
        )
    )
])

preprocessor_v2 = ColumnTransformer([
    ('categorical', categorical_pipeline_v2, categorical_cols_v2),
    ('numerical', numerical_pipeline_v2, numerical_cols_v2)
])


## 4. Model & Cross-validation


In [17]:
model_v2 = LGBMRegressor(
    objective='regression_l1',
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

pipeline_v2 = Pipeline([
    ('preprocessor', preprocessor_v2),
    ('model', model_v2)
])


In [18]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

v2_mae = -cross_val_score(
    pipeline_v2,
    X_v2,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

V1_CV_MAE = 0.211783

print('V2 Fold MAE:', np.round(v2_mae, 6))
print('V1 Mean MAE:', V1_CV_MAE)
print('V2 Mean MAE:', round(v2_mae.mean(), 6))
print('MAE Change:', round(v2_mae.mean() - V1_CV_MAE, 6))


V2 Fold MAE: [0.200132 0.203161 0.210994 0.218169 0.209165]
V1 Mean MAE: 0.211783
V2 Mean MAE: 0.208324
MAE Change: -0.003459


## 5. Final Training & Prediction


In [19]:
pipeline_v2.fit(X_v2, y)

pred_v2 = pipeline_v2.predict(X_test_v2)
pred_v2 = np.clip(pred_v2, 0, 1)

print('Prediction shape:', pred_v2.shape)
print('Prediction range:', pred_v2.min(), pred_v2.max())


Prediction shape: (3000,)
Prediction range: 0.12351552574181261 0.9112274690382898


## 6. Submission


In [20]:
submission_v2 = submission.copy()
submission_v2[ID_COL] = test_ids
submission_v2[TARGET] = pred_v2

output_path_v2 = DATA_DIR.parent / 'submit_v2.csv'
submission_v2.to_csv(output_path_v2, index=False)

print('Saved:', output_path_v2)
submission_v2.head()


Saved: /Users/bitsaem/Desktop/stress_project/submit_v2.csv


,ID,stress_score
0,TEST_0000,0.394572
1,TEST_0001,0.627595
2,TEST_0002,0.267760
3,TEST_0003,0.451427
4,TEST_0004,0.542802
